# CBraMod: дообучение на NMT (Пакистан) → проверка на турецком наборе

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-20 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Проверить базовую модель CBraMod по той же схеме, что DeepEpilepsy: дообучение на NMT («норма / патология») → проверка на турецком наборе («эпилепсия / здоровые»), **не виденном при обучении** |
| **Данные** | Обучение: весь NMT train (Kaggle `mhmdrdwn/nmt-eeg`), как в DeepEpilepsy v2. Проверка: NMT eval и Turkish Epilepsy EEG (Kaggle `buraktaci/turkish-epilepsy`), 50 эпилепсия + 71 здоровый |
| **Модель** | CBraMod (ICLR 2025), веса HF `weighting666/CBraMod` `pretrained_weights.pth`, 4,9 млн параметров; голова — двухслойная по всем патчам (как `all_patch_reps_twolayer` у авторов для TUAB/TUEV) |
| **Подготовка (по коду авторов)** | 19 каналов в порядке предобучения FP1 FP2 F3 F4 C3 C4 P3 P4 O1 O2 F7 F8 T3 T4 T5 T6 FZ CZ PZ; средний референт; 200 Гц; фильтр 0,3–75 Гц + режекция сети (у авторов 60 Гц для США, у нас **50 Гц**); окно 30 с → (19, 30, 200); **мкВ / 100** (так в `pretrain_trainer.py` и `tuab_dataset.py`). **Поправка масштаба NMT**: в EDF NMT амплитуда занижена примерно в 100 раз (СКО ≈0,2 «мкВ»), поэтому весь набор умножается на один множитель, чтобы медианное СКО стало 15 мкВ (метки не используются) |
| **Версия 2** | Первый запуск (без поправки масштаба) не сошёлся: модель выдавала почти константу, Турция 0,34. Причина — почти нулевой вход на NMT. Исправлено: поправка масштаба, быстрый фильтр, чтение в 4 потока |
| **Обучение** | 3 запуска (начальные значения 0, 1, 2), выбор эпохи по 15 % записей NMT train, классы уравновешены, до 8 окон на запись |
| **Среда** | Kaggle, GPU T4, Internet On |

Задачи разные: на NMT — «патология», на турецком — «эпилепсия». У многих больных эпилепсией рутинная ЭЭГ между приступами нормальна, поэтому очень высоких оценок не ждём.

Сравнение: DeepEpilepsy v2 по той же схеме (всё NMT, 3 запуска, Kaggle `zakharsann/h1-deepepilepsy-v2`): ансамбль 3 запусков — NMT eval 0,84–0,90, Турция 0,73 (95% ДИ 0,64–0,81), отдельные запуски на Турции 0,64–0,77; относительные мощности ритмов — Турция 0,66, NMT eval 0,83. Первая версия (подвыборка NMT, 1 запуск): Турция 0,66.

## 1. Модель

In [ ]:
import os, sys, glob, time, subprocess
import numpy as np, pandas as pd, torch, torch.nn as nn, mne, scipy.io, scipy.signal as ss
from huggingface_hub import hf_hub_download
from sklearn.metrics import roc_auc_score
mne.set_log_level("ERROR")

REPO = "/tmp/CBraMod"   # не в /kaggle/working, чтобы не попадало в выходные файлы
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/wjq-learning/CBraMod.git", REPO], check=True)
sys.path.insert(0, REPO)
from models import cbramod as cbm_mod
from models.cbramod import CBraMod

# Быстрое преобразование Фурье внутри PatchEmbedding не работает в fp16 при длине 200,
# поэтому этот блок всегда считаем в fp32, а трансформер — в fp16 (ускорение на T4).
_pe_forward = cbm_mod.PatchEmbedding.forward
def _pe_forward_fp32(self, x, mask=None):
    with torch.autocast("cuda", enabled=False):
        return _pe_forward(self, x.float(), mask)
cbm_mod.PatchEmbedding.forward = _pe_forward_fp32
AMP = dict(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available())

SFREQ, SEG, PATCH = 200, 30, 200
L = SFREQ * SEG
W_PATH = hf_hub_download("weighting666/CBraMod", "pretrained_weights.pth")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
PRETRAINED = torch.load(W_PATH, map_location="cpu", weights_only=True)

class Net(nn.Module):
    """CBraMod + голова по всем патчам (как all_patch_reps_twolayer у авторов)."""
    def __init__(self, n_ch=19, dropout=0.1):
        super().__init__()
        self.backbone = CBraMod(in_dim=200, out_dim=200, d_model=200, dim_feedforward=800, seq_len=30, n_layer=12, nhead=8)
        self.backbone.load_state_dict(PRETRAINED, strict=True)
        self.backbone.proj_out = nn.Identity()
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(n_ch * SEG * 200, 200), nn.ELU(), nn.Dropout(dropout), nn.Linear(200, 1))
    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(-1)

def new_model(seed):
    torch.manual_seed(seed)
    return Net().to(DEV)

def to_input(W):
    """(n, 19, 6000) в мкВ → (n, 19, 30, 200), деление на 100 как у авторов.
    Обрезка ±500 мкВ только против редких выбросов (при предобучении окна >100 мкВ вообще отбрасывались)."""
    x = np.clip(W.astype(np.float32) / 100.0, -5, 5)
    return torch.from_numpy(x.reshape(len(x), 19, SEG, PATCH))

def predict(model, Ws, bs=64):
    """Медиана вероятности по окнам для каждой записи."""
    model.eval(); out = []
    with torch.no_grad(), torch.autocast(**AMP):
        for W in Ws:
            p = [torch.sigmoid(model(to_input(W[b:b + bs]).to(DEV)).float()).cpu().numpy() for b in range(0, len(W), bs)]
            out.append(np.median(np.concatenate(p)))
    return np.array(out)

def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

BANDS = [(1, 4), (4, 8), (8, 13), (13, 30), (30, 45)]
def band_powers(W):
    fr, P = ss.welch(W, fs=SFREQ, nperseg=400, axis=-1)
    bp = np.stack([P[..., (fr >= a) & (fr < b)].mean(-1) for a, b in BANDS], -1)
    return np.concatenate([np.log(bp + 1e-12).mean(0).ravel(), (bp / bp.sum(-1, keepdims=True)).mean(0).ravel()])

SOS_BP = ss.butter(4, [0.3, 75], btype="bandpass", fs=SFREQ, output="sos")
B_N, A_N = ss.iirnotch(50, 30, fs=SFREQ)
def cbramod_filter(X):
    """Полоса как при предобучении CBraMod: 0,3–75 Гц + режекция сети (50 Гц для Пакистана и Турции). X: (19, t), 200 Гц.
    Баттерворт 4-го порядка в обе стороны вместо КИХ-фильтра MNE — в разы быстрее, полоса та же."""
    if X.shape[-1] < 60 * SFREQ:          # запись короче минуты — окон всё равно почти нет
        return X
    X = ss.sosfiltfilt(SOS_BP, X.astype(np.float64), axis=-1)
    return ss.filtfilt(B_N, A_N, X, axis=-1)

def windows(X):
    n = X.shape[1] // L
    return X[:, :n * L].reshape(19, n, L).transpose(1, 0, 2)

m = new_model(0)
print("устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор",
      "| параметров:", sum(p.numel() for p in m.parameters()), "| выход на случайных данных:", m(torch.randn(2, 19, 30, 200).to(DEV)).shape)
del m

## 2. NMT: чтение обучающей части (до 8 окон по 30 с на запись) и проверочной (все окна)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

ROOT_NMT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
lab = pd.read_csv(f"{ROOT_NMT}/Labels.csv")
paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT_NMT}/**/*.edf", recursive=True)}
lab["path"] = lab["recordname"].map(paths)
lab = lab.dropna(subset=["path"]).reset_index(drop=True)
lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)

# всё NMT (train + eval), как в DeepEpilepsy v2
lab = lab.sort_values("recordname").reset_index(drop=True)
print(pd.crosstab(lab["loc"], lab["label"]))

# параллельная подкачка файлов в кэш (32 потока)
def warm(p):
    with open(p, "rb") as f:
        while f.read(1 << 24):
            pass
t0 = time.time()
with ThreadPoolExecutor(32) as ex:
    for i, _ in enumerate(ex.map(warm, lab.path)):
        if (i + 1) % 200 == 0 or i + 1 == len(lab):
            print(f"подкачано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)

# порядок каналов, как при предобучении CBraMod (preprocessing_tueg_for_pretraining.py)
ELECTRODES = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]
ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
TARGET = [e.upper() for e in ELECTRODES]
def norm(ch):
    c = ch.upper().replace("EEG", "").strip()
    for s in ["-REF", "-AVG", "-LE", "-AR"]:
        c = c.replace(s, "")
    return ALIAS.get(c.strip(" -_"), c.strip(" -_"))

def load_nmt(path, max_win=None, rng=None):
    raw = mne.io.read_raw_edf(path, preload=True)
    if raw.info["sfreq"] != SFREQ:
        raw.resample(SFREQ)
    names = {norm(c): c for c in raw.ch_names}
    X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6         # мкВ, средний референт (как в NMT)
    W_raw, W = windows(X), windows(cbramod_filter(X))
    if max_win and len(W) > max_win:
        k = np.sort(rng.choice(len(W), max_win, replace=False))
        W_raw, W = W_raw[k], W[k]
    return W, W_raw

raw0 = mne.io.read_raw_edf(lab.path[0], preload=False)
print("единицы в заголовке EDF NMT:", sorted(set(raw0._orig_units.values())) if hasattr(raw0, "_orig_units") else "нет данных")

def nmt_job(i):
    r = lab.iloc[i]
    try:
        W, W_raw = load_nmt(r.path, max_win=None if r["loc"] == "eval" else 8, rng=np.random.default_rng(i))
    except Exception:
        return r.recordname, None, None
    if len(W) == 0:
        return r.recordname, None, None
    return r.recordname, W.astype(np.float32), band_powers(W_raw)   # мощности — как в прошлых блокнотах, без фильтра и пересчёта

NMT_W, NMT_BP = {}, {}
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, (name, W, bp) in enumerate(ex.map(nmt_job, range(len(lab)))):
        if W is not None:
            NMT_W[name], NMT_BP[name] = W, bp
        if (i + 1) % 400 == 0 or i + 1 == len(lab):
            print(f"NMT прочитано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)
lab = lab[lab.recordname.isin(NMT_W)].reset_index(drop=True)

# Масштаб. В первом запуске СКО сигнала NMT после get_data()·1e6 вышло ≈0,2 мкВ (у турецкого набора 14 мкВ,
# у обычной ЭЭГ — порядка 10–20 мкВ): физические единицы в EDF NMT, по-видимому, записаны неверно, и CBraMod получал почти ноль.
# DeepEpilepsy этого не замечала (нормирует каждое окно). Поправка — один множитель на весь набор, без меток:
# медианное СКО обучающих окон NMT приводим к 15 мкВ.
REF_UV = 15.0
med_std = float(np.median([NMT_W[n].std() for n in lab.recordname[lab["loc"] == "train"]]))
NMT_K = REF_UV / med_std
print(f"медианное СКО NMT до поправки {med_std:.3g} мкВ → множитель {NMT_K:.3g}")
for n in NMT_W:
    NMT_W[n] = (NMT_W[n] * NMT_K).astype(np.float16)
tr, ev = lab[lab["loc"] == "train"].reset_index(drop=True), lab[lab["loc"] == "eval"].reset_index(drop=True)
print(pd.crosstab(lab["loc"], lab["label"]))
print("доля отсчётов |x| > 100 мкВ в обучающих окнах NMT:",
      round(float(np.mean([np.mean(np.abs(NMT_W[n].astype(np.float32)) > 100) for n in tr.recordname])), 4))

## 3. Турецкий набор: 19 каналов в порядке CBraMod, **средний референт**, 200 Гц, фильтр 0,3–75 Гц + 50 Гц, окна 30 с

Куски по 15 с идут подряд (проверено в блокноте `h1_2026-10-18_zs_deepepilepsy-turkish`), поэтому склеиваем их в одну запись, фильтруем целиком и режем на окна по 30 с (= два соседних куска, как раньше).

In [ ]:
ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
         [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]
t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    for i, _ in enumerate(ex.map(warm, [f for f, _ in tfiles])):
        if (i + 1) % 20 == 0 or i + 1 == len(tfiles):
            print(f"Турция подкачано {i+1}/{len(tfiles)} {time.time()-t0:.0f} с", flush=True)
# индексы турецких каналов в порядке CBraMod:   Fp1 Fp2 F3 F4 C3 C4 P3 P4 O1  O2  F7  F8  T3  T4  T5  T6  Fz Cz Pz
IDX = [0, 1, 2, 3, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 4, 7, 10]

def tr_job(f):
    d = scipy.io.loadmat(f)["data"]
    X = np.stack([np.stack([d[s, c].ravel() for c in IDX]) for s in range(d.shape[0])])   # (куски, 19, 7500)
    X = X - X.mean(1, keepdims=True)                                                       # средний референт
    X = ss.resample_poly(X, 2, 5, axis=-1)                                                  # 200 Гц, (куски, 19, 3000)
    n = len(X) // 2
    W_raw = np.concatenate([X[0:2*n:2], X[1:2*n:2]], axis=-1)                              # (окна, 19, 6000), как раньше
    Xc = np.concatenate(list(X), axis=-1)                                                   # (19, куски·3000), запись целиком
    return windows(cbramod_filter(Xc)).astype(np.float16), band_powers(W_raw)

TR_W, TR_BP = [], []
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, (W, bp) in enumerate(ex.map(tr_job, [f for f, _ in tfiles])):
        TR_W.append(W); TR_BP.append(bp)
        if (i + 1) % 20 == 0 or i + 1 == len(tfiles):
            print(f"Турция {i+1}/{len(tfiles)} {time.time()-t0:.0f} с", flush=True)
y_tr = np.array([yy for _, yy in tfiles])
print("эпилепсия:", int(y_tr.sum()), "| здоровые:", int((1 - y_tr).sum()),
      "| окон на запись: медиана", int(np.median([len(w) for w in TR_W])))
print("размах сигнала (СКО, мкВ): NMT train", round(float(np.median([NMT_W[n].astype(np.float32).std() for n in tr.recordname])), 1),
      "| Турция", round(float(np.median([w.astype(np.float32).std() for w in TR_W])), 1))

## 4. Дообучение CBraMod на NMT train, 3 запуска

Как у авторов для TUAB: AdamW, lr основы 1·10⁻⁴, lr головы 5·10⁻⁴ (= 0,001·√(64/256)), weight decay 0,05, косинусное снижение до 10⁻⁶, BCE, обрезка градиента 1, пакет 64, dropout 0,1. У авторов 50 эпох — у нас **6** (ограничение по времени GPU; если 3 запуска не укладываются в 50 мин, лишние пропускаются). Классы уравновешены выборкой окон с весами, до 8 окон на запись. Эпоха выбирается по ROC AUC на 15 % записей NMT train (деление одинаковое во всех запусках, меняется начальное значение). Турецкий набор при обучении и выборе эпохи не используется. Трансформер считается в fp16, спектральный блок входа — в fp32 (быстрое преобразование Фурье не работает в fp16 при длине 200).

In [ ]:
from sklearn.model_selection import train_test_split

fit_names, val_names = train_test_split(tr.recordname.to_numpy(), test_size=0.15, stratify=tr.y, random_state=0)
ymap = dict(zip(lab.recordname, lab.y))
Xw = np.concatenate([NMT_W[n] for n in fit_names])
yw = np.concatenate([np.full(len(NMT_W[n]), ymap[n]) for n in fit_names])
y_val = np.array([ymap[n] for n in val_names])
print("окон для обучения:", len(yw), "| из них патология:", int(yw.sum()), "| записей для выбора эпохи:", len(val_names))

EPOCHS, BS, SEEDS = 6, 64, [0, 1, 2]
TIME_LIMIT_MIN = 50   # ограничение на всё обучение, чтобы не выйти за ~2 ч GPU вместе с чтением данных
w = np.where(yw == 1, 0.5 / (yw == 1).sum(), 0.5 / (yw == 0).sum())
steps = int(np.ceil(len(yw) / BS))
RES = []
T_START = time.time()
for seed in SEEDS:
    if RES and (time.time() - T_START) / 60 * (len(RES) + 1) / len(RES) > TIME_LIMIT_MIN:
        print(f"останов по времени: следующий запуск не уложится в {TIME_LIMIT_MIN} мин, выполнено запусков: {len(RES)}"); break
    model = new_model(seed)
    back = [p for n, p in model.named_parameters() if n.startswith("backbone")]
    other = [p for n, p in model.named_parameters() if not n.startswith("backbone")]
    opt = torch.optim.AdamW([{"params": back, "lr": 1e-4}, {"params": other, "lr": 1e-3 * (BS / 256) ** 0.5}], weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * steps, eta_min=1e-6)
    lossf = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    best_auc, best_state, best_ep, n_bad = -1, None, 0, 0
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train()
        idx = np.random.default_rng(1000 * seed + ep).choice(len(yw), size=len(yw), replace=True, p=w / w.sum())
        losses = []
        for b in range(0, len(idx), BS):
            j = idx[b:b + BS]
            x = to_input(Xw[j]).to(DEV)
            t = torch.tensor(yw[j], dtype=torch.float32, device=DEV)
            with torch.autocast(**AMP):
                logit = model(x)
            loss = lossf(logit.float(), t)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); losses.append(loss.item())
            if not np.isfinite(losses[-1]): n_bad += 1
        auc = roc_auc_score(y_val, predict(model, [NMT_W[n] for n in val_names]))
        if auc > best_auc:
            best_auc, best_ep, best_state = auc, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"запуск {seed} | эпоха {ep+1}: потери {np.mean(losses):.3f} | ROC AUC на отложенных записях NMT {auc:.3f} | шагов с NaN/inf {n_bad} | {time.time()-t0:.0f} с", flush=True)
    model.load_state_dict(best_state)
    s_nmt = predict(model, [NMT_W[n] for n in ev.recordname])
    s_tr = predict(model, TR_W)
    RES.append(dict(seed=seed, ep=best_ep, val=best_auc, nmt=roc_auc_score(ev.y, s_nmt), tr=roc_auc_score(y_tr, s_tr), s_nmt=s_nmt, s_tr=s_tr))
    print(f"запуск {seed}: эпоха {best_ep} | NMT eval {RES[-1]['nmt']:.3f} | Турция {RES[-1]['tr']:.3f}", flush=True)
    del model, opt; torch.cuda.empty_cache()
print(f"всё обучение: {(time.time()-T_START)/60:.1f} мин")

## 5. Главное: CBraMod, дообученная на NMT → NMT eval и турецкий набор

In [ ]:
rows = []
for r in RES:
    lo, hi = boot_ci(y_tr, r["s_tr"])
    rows.append({"запуск": r["seed"], "эпоха": r["ep"], "NMT отлож.": round(r["val"], 3), "NMT eval": round(r["nmt"], 3),
                 "Турция AUC": round(r["tr"], 3), "Турция 95% ДИ": f"{lo:.2f}–{hi:.2f}",
                 "медиана p: здоровые": round(float(np.median(r["s_tr"][y_tr == 0])), 3),
                 "медиана p: эпилепсия": round(float(np.median(r["s_tr"][y_tr == 1])), 3)})
# ансамбль запусков: среднее рангов предсказаний (как в DeepEpilepsy v2)
from scipy.stats import rankdata
def rank(s): return rankdata(s) / len(s)
s_ens_tr = np.mean([rank(r["s_tr"]) for r in RES], 0); s_ens_nmt = np.mean([rank(r["s_nmt"]) for r in RES], 0)
lo, hi = boot_ci(y_tr, s_ens_tr)
rows.append({"запуск": f"ансамбль {len(RES)} (ранги)", "эпоха": "—", "NMT отлож.": "—", "NMT eval": round(roc_auc_score(ev.y, s_ens_nmt), 3),
             "Турция AUC": round(roc_auc_score(y_tr, s_ens_tr), 3), "Турция 95% ДИ": f"{lo:.2f}–{hi:.2f}",
             "медиана p: здоровые": "—", "медиана p: эпилепсия": "—"})
print(pd.DataFrame(rows).to_string(index=False))
nm, tu = np.array([r["nmt"] for r in RES]), np.array([r["tr"] for r in RES])
print(f"\nСреднее ± разброс (СКО) по {len(RES)} запускам: NMT eval {nm.mean():.3f} ± {nm.std(ddof=1):.3f} | Турция {tu.mean():.3f} ± {tu.std(ddof=1):.3f}")
lo_ev, hi_ev = boot_ci(ev.y.to_numpy(), s_ens_nmt)
print(f"NMT eval, ансамбль: 95% ДИ {lo_ev:.2f}–{hi_ev:.2f} (записей {len(ev)})")

## 6. Простая модель для сравнения: мощности ритмов, обучение на NMT train → турецкий набор (как в блокноте DeepEpilepsy)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

Xb_nmt = np.stack([NMT_BP[n] for n in tr.recordname]); Xb_tr = np.stack(TR_BP)
half = Xb_nmt.shape[1] // 2
BP_S = {}
for name, sl in [("все мощности", slice(None)), ("только относительные (без амплитуды)", slice(half, None))]:
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=5000, class_weight="balanced"))
    s_bp = clf.fit(Xb_nmt[:, sl], tr.y).predict_proba(Xb_tr[:, sl])[:, 1]
    BP_S[name] = s_bp
    lo, hi = boot_ci(y_tr, s_bp)
    print(f"Мощности ритмов, {name:38s}: ROC AUC = {roc_auc_score(y_tr, s_bp):.3f} (95% ДИ {lo:.2f}–{hi:.2f})")
from scipy.stats import spearmanr
print("связь оценок CBraMod (ансамбль) и относительных мощностей на Турции, Спирмен:",
      round(spearmanr(s_ens_tr, BP_S["только относительные (без амплитуды)"]).correlation, 2))

# сводка, как в DeepEpilepsy v2: ансамбль запусков, ансамбль с относительными мощностями, сами мощности
clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=5000, class_weight="balanced")).fit(Xb_nmt[:, half:], tr.y)
bp_tr = clf.predict_proba(Xb_tr[:, half:])[:, 1]
bp_ev = clf.predict_proba(np.stack([NMT_BP[n] for n in ev.recordname])[:, half:])[:, 1]
rows = []
for title, se, st in [(f"CBraMod, ансамбль {len(RES)} запусков", s_ens_nmt, s_ens_tr),
                      ("CBraMod + относит. мощности", (s_ens_nmt + rank(bp_ev)) / 2, (s_ens_tr + rank(bp_tr)) / 2),
                      ("Относительные мощности ритмов", bp_ev, bp_tr)]:
    lo, hi = boot_ci(y_tr, st)
    rows.append(dict(модель=title, NMT_eval=round(roc_auc_score(ev.y, se), 3),
                     Турция=round(roc_auc_score(y_tr, st), 3), Турция_95ДИ=f"{lo:.2f}–{hi:.2f}"))
summary = pd.DataFrame(rows)
summary.to_csv("/kaggle/working/cbramod_summary.csv", index=False)
print(summary.to_string(index=False))

## 7. Результаты запуска (вывод Kaggle, версия 2)

Обучение: 3 запуска × 6 эпох, 44 мин на T4, шагов с NaN/inf нет. Тестовые данные (NMT eval, Турция) считались один раз, при отладке по ним ничего не подбиралось.

```
единицы в заголовке EDF NMT: ['µV']
медианное СКО NMT до поправки 0.189 мкВ → множитель 79.3
размах сигнала (СКО, мкВ): NMT train 15.0 | Турция 13.9

окон для обучения: 15007 | из них патология: 2169 | записей для выбора эпохи: 334
запуск 0 | эпоха 1: потери 0.629 | ROC AUC на отложенных записях NMT 0.846 | шагов с NaN/inf 0 | 136 с
запуск 0 | эпоха 2: потери 0.333 | ROC AUC на отложенных записях NMT 0.885 | шагов с NaN/inf 0 | 278 с
запуск 0 | эпоха 3: потери 0.168 | ROC AUC на отложенных записях NMT 0.856 | шагов с NaN/inf 0 | 420 с
запуск 0 | эпоха 4: потери 0.075 | ROC AUC на отложенных записях NMT 0.890 | шагов с NaN/inf 0 | 562 с
запуск 0 | эпоха 5: потери 0.036 | ROC AUC на отложенных записях NMT 0.878 | шагов с NaN/inf 0 | 704 с
запуск 0 | эпоха 6: потери 0.020 | ROC AUC на отложенных записях NMT 0.881 | шагов с NaN/inf 0 | 845 с
запуск 0: эпоха 4 | NMT eval 0.912 | Турция 0.689
запуск 1 | эпоха 1: потери 0.603 | ROC AUC на отложенных записях NMT 0.834 | шагов с NaN/inf 0 | 143 с
запуск 1 | эпоха 2: потери 0.320 | ROC AUC на отложенных записях NMT 0.876 | шагов с NaN/inf 0 | 286 с
запуск 1 | эпоха 3: потери 0.168 | ROC AUC на отложенных записях NMT 0.858 | шагов с NaN/inf 0 | 429 с
запуск 1 | эпоха 4: потери 0.072 | ROC AUC на отложенных записях NMT 0.865 | шагов с NaN/inf 0 | 571 с
запуск 1 | эпоха 5: потери 0.031 | ROC AUC на отложенных записях NMT 0.865 | шагов с NaN/inf 0 | 712 с
запуск 1 | эпоха 6: потери 0.018 | ROC AUC на отложенных записях NMT 0.860 | шагов с NaN/inf 0 | 853 с
запуск 1: эпоха 2 | NMT eval 0.895 | Турция 0.886
запуск 2 | эпоха 1: потери 0.635 | ROC AUC на отложенных записях NMT 0.814 | шагов с NaN/inf 0 | 143 с
запуск 2 | эпоха 2: потери 0.343 | ROC AUC на отложенных записях NMT 0.841 | шагов с NaN/inf 0 | 285 с
запуск 2 | эпоха 3: потери 0.175 | ROC AUC на отложенных записях NMT 0.861 | шагов с NaN/inf 0 | 428 с
запуск 2 | эпоха 4: потери 0.084 | ROC AUC на отложенных записях NMT 0.860 | шагов с NaN/inf 0 | 570 с
запуск 2 | эпоха 5: потери 0.037 | ROC AUC на отложенных записях NMT 0.856 | шагов с NaN/inf 0 | 711 с
запуск 2 | эпоха 6: потери 0.022 | ROC AUC на отложенных записях NMT 0.861 | шагов с NaN/inf 0 | 852 с
запуск 2: эпоха 3 | NMT eval 0.884 | Турция 0.669
всё обучение: 44.2 мин

            запуск эпоха NMT отлож.  NMT eval  Турция AUC Турция 95% ДИ медиана p: здоровые медиана p: эпилепсия
                 0     4       0.89     0.912       0.689     0.58–0.78                 0.0                  0.0
                 1     2      0.876     0.895       0.886     0.82–0.94                 0.0                0.003
                 2     3      0.861     0.884       0.669     0.56–0.77                 0.0                  0.0
ансамбль 3 (ранги)     —          —     0.908       0.783     0.70–0.86                   —                    —

Среднее ± разброс (СКО) по 3 запускам: NMT eval 0.897 ± 0.014 | Турция 0.748 ± 0.120
NMT eval, ансамбль: 95% ДИ 0.86–0.95 (записей 184)

Мощности ритмов, все мощности                          : ROC AUC = 0.719 (95% ДИ 0.61–0.81)
Мощности ритмов, только относительные (без амплитуды)  : ROC AUC = 0.636 (95% ДИ 0.53–0.74)
связь оценок CBraMod (ансамбль) и относительных мощностей на Турции, Спирмен: 0.51
```

Сводка (`results/cbramod_summary.csv`):

```
модель,NMT_eval,Турция,Турция_95ДИ
"CBraMod, ансамбль 3 запусков",0.908,0.783,0.70–0.86
CBraMod + относит. мощности,0.891,0.723,0.63–0.82
Относительные мощности ритмов,0.822,0.636,0.53–0.74
```

## 8. Выводы

### Сравнение (проверка на турецком наборе, не виденном при обучении)

| Модель | NMT eval | Турция, ROC AUC (95% ДИ) |
|---|---|---|
| **CBraMod, ансамбль 3 запусков** | **0,91 (0,86–0,95)** | **0,78 (0,70–0,86)** |
| CBraMod, отдельные запуски (среднее ± СКО) | 0,90 ± 0,01 | 0,75 ± 0,12 (0,69 / 0,89 / 0,67) |
| CBraMod + относительные мощности (ранги) | 0,89 | 0,72 (0,63–0,82) |
| DeepEpilepsy v2, ансамбль 3 запусков | 0,90 | 0,73 (0,64–0,81) |
| DeepEpilepsy v2, отдельные запуски | — | 0,72 ± 0,01 |
| Относительные мощности ритмов (обучение на NMT) | 0,82 | 0,64 (0,53–0,74) |
| Все мощности ритмов (с амплитудой) | — | 0,72 (0,61–0,81) |

1. **CBraMod после дообучения на NMT даёт на турецком наборе 0,78 (0,70–0,86) в ансамбле** — выше относительных мощностей ритмов (0,64) и в пределах доверительных интервалов близко к DeepEpilepsy v2 (0,73). Разница с DeepEpilepsy (0,05) меньше ширины интервалов, поэтому «лучше» утверждать нельзя; по NMT eval модели практически равны (0,91 и 0,90).
2. **Оценка нестабильна.** Отдельные запуски на Турции — от 0,67 до 0,89 (0,75 ± 0,12), тогда как на NMT eval они почти совпадают (0,90 ± 0,01). Без аугментаций результат на новом наборе сильно зависит от начального значения; ансамбль сглаживает разброс, но 0,89 у одного запуска случайность, а не свойство метода. Потери на обучении падают до 0,02, то есть модель запоминает обучающие окна; лучшие эпохи по отложенным записям — 2–4.
3. Вероятности насыщены (медианы у здоровых и больных ≈0), поэтому для ранжирования используем ранги, а как «вероятность эпилепсии» выход модели без калибровки не годится.

### Что менялось относительно первого запуска и почему

Первый запуск не сошёлся (потери ≈0,69, ROC AUC на отложенных записях NMT падал до 0,50, на тесте — NMT 0,66, Турция 0,34; результат недействителен). Причина: в файлах NMT масштаб неверный — после чтения СКО сигнала ≈0,19 мкВ, хотя в заголовке указаны мкВ (у реальной ЭЭГ ~10–20 мкВ, в турецком наборе — 14 мкВ). CBraMod получает сигнал в мкВ/100, поэтому на NMT видела почти ноль. Изменения во втором запуске:
- **масштаб NMT**: весь набор умножен на один множитель 79,3, чтобы медианное СКО обучающих окон стало 15 мкВ (метки не использовались; множитель посчитан только по NMT train; значение 15 взято из типичной ЭЭГ, не подбиралось);
- фильтр 0,3–75 Гц + режекция 50 Гц: КИХ-фильтр MNE заменён на фильтр Баттерворта 4-го порядка (в обе стороны) — тот же диапазон, но быстрее;
- чтение NMT и турецкого набора в 4 потока (ускорение, на результат не влияет);
- лимит времени обучения 80 → 50 мин (фактически 44 мин, все 3 запуска выполнены);
- скорость обучения, число эпох, точность вычислений и остальные настройки не менялись; добавлен подсчёт шагов с NaN/inf (их не было).

Проверка сходимости перед запуском — только на синтетических окнах (не на NMT eval и не на турецком наборе). Замечание: СКО турецкого сигнала (14 мкВ) было видно в логе первого запуска и могло повлиять на выбор «разумного» уровня 15 мкВ; сам множитель от турецких данных не зависит.

### Оговорки

1. **CBraMod получает сигнал в абсолютных мкВ** (без нормировки каждого окна, в отличие от DeepEpilepsy), а в турецком наборе у больных амплитуда выше (одна амплитуда там даёт 0,79; «все мощности» с амплитудой — 0,72). Часть оценки 0,78 может идти от этого перекоса, а не от рисунка ЭЭГ. Кроме того, масштаб NMT приведён общим множителем «на глаз», так что связь «амплитуда → патология» могла выучиться некорректно. Корреляция оценок CBraMod с относительными мощностями на Турции 0,51 (Спирмен): частично модель видит то же, что простые признаки. Нужна проверка с нормировкой каждого окна и с аугментацией масштаба амплитуды.
2. **Большой разброс между запусками** на Турции (0,67–0,89); ансамбль из 3 — всего 3 модели, сильное запоминание обучающих окон.
3. **CBraMod предобучена на TUEG** (69 652 записи TUH): для будущей проверки на TUEP это пересечение с тестовыми пациентами — понадобится исключить их по ID или не использовать эту модель для честной оценки на TUEP.
4. Задачи разные (NMT — «патология», Турция — «эпилепсия»); контроль в турецком наборе — здоровые; разная аппаратура, референт пересчитан к среднему; в сравнении DeepEpilepsy v2 получена при других условиях обучения (аугментации, нормировка окон).

### Затраты

GPU T4: первый запуск 88 мин, второй около 1,3 ч (44 мин обучения + чтение данных).

## 9. Что дальше

- CBraMod с нормировкой окна и аугментациями (шум, зануление каналов, сдвиг, масштаб амплитуды), 5 запусков вместо 3; ранняя остановка по 2–3 эпохе.
- Ансамбль CBraMod + DeepEpilepsy v2.
- Когда придёт доступ к TUH: NMT/TUAB → TUEP, с исключением пациентов TUEG, видевших предобучение CBraMod.